# Data Exploration

Quick overview of the raw UCF-101 / Kinetics-400 subset before curation.

In [ ]:
import sys
sys.path.insert(0, '../src')

import glob
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

sns.set_theme(style='whitegrid')
RAW_DIR = Path('../data/raw/ucf101')

In [ ]:
# Find all clips
clips = list(RAW_DIR.rglob('*.avi')) + list(RAW_DIR.rglob('*.mp4'))
print(f'Found {len(clips)} clips in {RAW_DIR}')

# Class distribution
labels = [p.parent.name for p in clips]
label_counts = Counter(labels)

print(f'\nClasses ({len(label_counts)}):')
for lbl, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {lbl:25s}: {cnt:4d} clips')

In [ ]:
# Duration distribution (sample first 100 clips)
sample_clips = clips[:100]
durations = []
for p in sample_clips:
    cap = cv2.VideoCapture(str(p))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    durations.append(n / fps)

plt.figure(figsize=(8, 4))
plt.hist(durations, bins=20, edgecolor='white')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.title(f'Clip Duration Distribution (n={len(durations)})')
plt.axvline(np.mean(durations), color='red', linestyle='--', label=f'Mean={np.mean(durations):.1f}s')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Mean: {np.mean(durations):.2f}s, Median: {np.median(durations):.2f}s')

In [ ]:
# Blur and motion score distribution on a small sample
from video_curation.curation.quality_filter import score_clip as quality_score
from video_curation.curation.motion_score import score_clip as motion_score

sample = [str(p) for p in clips[:50]]
blur_scores = []
mot_scores = []
sample_labels = []

for p in sample:
    try:
        qs = quality_score(p, blur_threshold=0)  # threshold=0 → never reject
        ms = motion_score(p, min_motion=0, max_motion=1e9)
        blur_scores.append(qs.blur_score)
        mot_scores.append(ms.mean_flow_magnitude)
        sample_labels.append(Path(p).parent.name)
    except Exception as e:
        print(f'  Error on {p}: {e}')

df_scores = pd.DataFrame({'label': sample_labels, 'blur': blur_scores, 'motion': mot_scores})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_scores, x='label', y='blur', ax=axes[0])
axes[0].set_title('Blur Score (Laplacian σ) by Class')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df_scores, x='label', y='motion', ax=axes[1])
axes[1].set_title('Motion Score by Class')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()